In [12]:
# importando os scripts de consulta

import sys
import os

# Altera o caminho para a pasta raiz do projeto (um nível acima)
raiz_projeto = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Adiciona a raiz do projeto ao caminho de busca do Python se ainda não estiver lá
if raiz_projeto not in sys.path:
    sys.path.append(raiz_projeto)

    
import pandas as pd
from scripts.crm_avaliacoes import load_avaliacoes
from scripts.crm_leads_escolas_parceiras import load_leads

### Carregamento dos leads

In [13]:
leads = load_leads()

Página 1: 8 registros


In [14]:
leads.columns

Index(['Total Caracteres Redação', 'crmeduc_redacaotrivento', 'Nota Redação',
       'Escola Lorena', 'Inscrição Atual', 'cpf', 'Tabulação Realizada',
       'Nome Completo', 'Nome', 'Sobrenome', 'Email', 'Celular', 'ID Lead',
       'Data Criação', 'Idade', 'Concurso Atual', 'Concurso de Origem',
       'Data Nascimento', 'Oferta Atual', 'Situação Funil', 'Unidade Lead',
       'Linha de Negócio', 'Proprietário', 'Unidade Form'],
      dtype='str')

In [15]:
colunas = ['ID Lead', 'Nome Completo', 'Inscrição Atual', 'Concurso Atual', 'Escola Lorena', 'Idade','Nota Redação', 'Total Caracteres Redação' ]

leads = leads[colunas]

In [16]:
leads['Idade'].describe()

count      8
unique     1
top       27
freq       8
Name: Idade, dtype: object

In [17]:
df = leads.copy()

In [18]:
'''Critérios de classificação dos candidatos:

Caso 1
Caso sua escola tenha 10 ou mais candidatos,  o candidato é rankeado dentro de sua escola


Caso 2
Caso sua escola menos que  10 candidatos,  o candidato é rankeado como Geral

Critérios:
- maior nota final
- maior nota de redação
- maior idade


'''

'Critérios de classificação dos candidatos:\n\nCaso 1\nCaso sua escola tenha 10 ou mais candidatos,  o candidato é rankeado dentro de sua escola\n\n\nCaso 2\nCaso sua escola menos que  10 candidatos,  o candidato é rankeado como Geral\n\nCritérios:\n- maior nota final\n- maior nota de redação\n- maior idade\n\n\n'

### Classificação

In [19]:

colunas_notas = [
    'Idade',
    'Nota Redação',
    'Total Caracteres Redação'
]

df_classificacao = df.copy()
df_classificacao[colunas_notas] = df_classificacao[colunas_notas].apply(
    pd.to_numeric, errors='coerce'
)
df_classificacao[colunas_notas] = df_classificacao[colunas_notas].fillna(0)

df_classificacao['Quantidade na escola'] = (
    df_classificacao.groupby('Escola Lorena')['ID Lead'].transform('size')
)
df_classificacao['Grupo de classificação'] = df_classificacao['Escola Lorena'].where(
    df_classificacao['Quantidade na escola'].ge(10),
    'Geral',
)

df_classificacao = df_classificacao.sort_values(
    by=[
        'Nota Redação',
        'Idade',
        'Total Caracteres Redação'
    ],
    ascending=[False,False, False],
    kind='stable',
).reset_index(drop=True)

df_classificacao['Classificação'] = (
    df_classificacao.groupby('Grupo de classificação').cumcount().add(1)
)

colunas_resultado = [
    'Classificação',
    'Grupo de classificação',
    'Quantidade na escola',
    'Escola Lorena',
    'Nome Completo',
    'Inscrição Atual',
    'Nota Redação',
    'Total Caracteres Redação',
    'Idade',
    'ID Lead',
]
resultado_classificacao = df_classificacao[colunas_resultado]
resultado_classificacao.to_csv(
    'classificacao_candidatos.csv',
    index=False,
    sep=';',
    encoding='utf-8-sig',
)

resultado_classificacao


,Classificação,Grupo de classificação,Quantidade na escola,Escola Lorena,Nome Completo,Inscrição Atual,Nota Redação,Total Caracteres Redação,Idade,ID Lead
0,1,Geral,4,Barão Da Bocaina - Areias,Hugo Oito,CAN-007355-TRI,90.0,2144.0,27,{E06B4D96-66E3-41A7-9AE3-7AE822074D02}
1,2,Geral,4,Barão Da Bocaina - Areias,Hgo Seis Teste,CAN-007348-TRI,85.0,546.0,27,{C362556F-9099-4DFE-905D-473A5BB35E77}
2,3,Geral,1,Cemi- Unidade 3 - Taubaté,Hugo Sete Teste,CAN-007353-TRI,10.0,798.0,27,{D98C2E57-8818-4629-903C-F26B87013C08}
3,4,Geral,2,Basic Taubaté - Taubaté,Hugo Teste Cinco,CAN-007345-TRI,0.0,3050.0,27,{F27D35BE-FBCF-43A8-A36E-69693A15546B}
4,5,Geral,2,Basic Taubaté - Taubaté,Hugoo Quatro,NaN,0.0,3050.0,27,{2F2FD9B6-F6E2-47BB-AFFC-DD7FD25A9678}
5,6,Geral,4,Barão Da Bocaina - Areias,Hugo Escola Teste,CAN-007350-TRI,0.0,0.0,27,{4ADE5131-CFF4-41A4-A6F3-CE83063FC3DF}
6,7,Geral,1,Canção Nova - Cachoeira Paulista,Hugo Teste Dois,CAN-007317-TRI,0.0,0.0,27,{DFCD8CFE-92CD-45C7-81E8-CA54760337DE}
7,8,Geral,4,Barão Da Bocaina - Areias,Hugo Teste,CAN-007315-TRI,0.0,0.0,27,{933265CA-669D-4706-9F7A-8EC49A94B53E}
